# nano Beatrix — the scale test's cores: Castor, and Alhena, the third of the triplets

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_scale_cores.ipynb)

**What trains**: seven cores of the nano body at the bed's recipe (12,000 steps of 8 rows x 4,096 bytes, pure Adam, two seeds), in the order each one gates the next:

| part | body | rows | what it is |
|---|---|---|---|
| R | Castor (`castor-both`) | the bed's framed rows | the stored 4,096 framed cell trained again verbatim (the replication arm); seed 1 trains twice (the hardware repeat) |
| H | Castor | half the rows addressed grid rows, the per-digit address head | the system without its rotary |
| A | **Alhena** (`alhena-both`) | the same | Castor's layout turned by the anchored multi-position rotary: the system |

**Alhena** is the third of the triplets: gamma Geminorum, the third-brightest star of the Twins after Pollux and Castor. Its name, al-han'a, is the mark: here, the address mark each compartment of the window carries. An addressed grid row is 32 slots of 128 bytes: a head, 24 compartments each holding one addressed lookup frame, an index the model writes (each asked key with its compartment's address), and 24 questions. Alhena's softmax blocks split each head's 16 rotate-half pairs by job: six pairs turn by the relative position (NTK-aware x4), two carry content unturned, eight turn by the two address digits a token sits in (keys) or asks for (the questions' answer positions).

**The cells**: C1 pins the trainer and holds the decision block; C2 builds the data, prints the plan, runs 200 steps of Alhena on this card for the price line and probes the push path; C3 trains the parts in order, every finished run shipped to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) (`bed/reports/` and `bed/weights/<run>/`, the address head and `scale.json` beside the weights); C4 renders the table.

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. A file named `STOP` in `/content/out` ends the session between runs; a restarted session resumes where it stopped. The trainer and its tests live in [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.70"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 70), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner ships through the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run ships here (its record and its weights); tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed: 160 MB covers 12,000 steps x 8 x 4,096 four times over
BED = dict(                                            # THE STORED 4,096 FRAMED CELL, verbatim (the replication arm's recipe)
    ctx=4096, mb=8, steps=12000,                       # 12,000 steps x 8 rows x 4,096 bytes; pure Adam, lr 1e-3, 100 warm-up steps, the cosine to a tenth
    frames='dyn', plant=1.0,                           # every row framed: the dynamic frames of generic sizes
    sizes='32,128,256,512', frame_records=4, nest=0.3, overlap=0.0, ask='any', register='caption', tags='mode',
    log_every=100,                                     # the trainer's progress line feeds the step bar
)
GRID_ROWS = dict(grid='ag', grid_p=0.5,                # half of every batch's rows are addressed grid rows (the other half the bed's rows, place for place)
                 head='digit', head_lam=1.0,           # the per-digit address head trained with the core (its novelty gated at step 0)
                 trace_every=2000)                     # the formation trace: the index, the answers and the head every 2,000 steps
PARTS = [                                              # the gate order: each seed's replication arm, then the system without its rotary, then the system
    ('R s1', dict(BED, arms='castor-both', seeds='1', series='scale')),
    ('H s1', dict(BED, **GRID_ROWS, arms='castor-both', seeds='1', series='scale')),
    ('Alhena s1', dict(BED, **GRID_ROWS, arms='alhena-both', seeds='1', series='scale')),
    ('R s2', dict(BED, arms='castor-both', seeds='2', series='scale')),
    ('H s2', dict(BED, **GRID_ROWS, arms='castor-both', seeds='2', series='scale')),
    ('Alhena s2', dict(BED, **GRID_ROWS, arms='alhena-both', seeds='2', series='scale')),
    ('R s1 again', dict(BED, arms='castor-both', seeds='1', series='scalerep')),   # the hardware repeat: the noise bar is max(this repeat, the seed spread)
]
SMOKE_STEPS = 200                     # C2's short run of Alhena on this card for the price line; its push is off
PARALLEL = 1                          # ONE run at a time: a 9M craft at 8 x 4,096 saturates the RTX PRO 6000 alone (863k tok/s); four at once ran at
                                      # ~214k each, their sum one run's rate: spare VRAM is not spare compute
print('decision block:', len(PARTS), 'parts | out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, 200 steps of Alhena on this card (the price line), the push probe. No part trains.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner

# the data: the corpus and the held-out head from fineweb-edu (tokenless; a few minutes; the lines carry the time left)
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'))

# the plan: every part's run and what this folder already holds
todo = 0
for name, spec in PARTS:
    p = bed_runner.run(spec, out=OUT, dry=True)
    todo += p['runs'] - p['done_before']

# the price line: Alhena (the grid rows, the rotation and the head: the dearest part) for SMOKE_STEPS on this card; the record stays local
smoke = dict(PARTS[2][1], steps=SMOKE_STEPS, log_every=20, trace_every=100, read_batches=4, calib_batches=2)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any part'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_run = BED['steps'] * BED['ctx'] * BED['mb'] / (last['tok_s'] * 1e3) / 60
print(f"this card: {last['tok_s']}k tok/s at {BED['ctx']} x {BED['mb']} -> one {BED['steps']:,}-step run about {per_run:.0f} min; the {todo} runs left about {per_run * todo / 60:.1f} h (+ ~1 min of end reads a run)")

# the push probe (the ship-complete law): a write to the training repo and a read-back BEFORE any training. Tokenless = disarmed.
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_scale.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'parts': [n for n, _ in PARTS]}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano scale cores: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')

In [ ]:
# C3 (T) — THE SESSION: the parts in gate order, one run at a time, resume-first (runner_done.txt in OUT); a STOP file in OUT starts no
# new run; every finished run ships to REPO as it lands (one background shipper for the whole session: the card never waits on the Hub).
# An interrupt loses only the running run (the bed has no mid-run checkpoint); rerun the cell to continue.
from geolip.nano_beatrix import bed_runner
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = bed_runner._Shipper(push, OUT, print) if push else None
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
done = []
for name, spec in PARTS:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C4 — THE TABLE: every finished part's held-out bits, the frame copy by size, and the grid read (the index, the answers by request, the head).
# Run any time; the Hub holds the same records under bed/reports/.
import glob, json
from IPython.display import Markdown, display
rows = ['| run | held-out bits | copy 32 / 128 / 256 / 512 | index right | answers whole | own index | null request | bank off | head right | minutes |',
        '|---|---|---|---|---|---|---|---|---|---|']
for p in sorted(glob.glob(f'{OUT}/bed/bed_*_L12000_*scale*.json')):
    r = json.load(open(p, encoding='utf-8'))
    ff = r.get('fetch_by_frame') or {}
    copy = ' / '.join(f"{ff[f'data/{s}/depth0']['full_value']:.3f}" if f'data/{s}/depth0' in ff else '-' for s in (32, 128, 256, 512))
    g = (r.get('grid') or {}).get('read') or {}
    f = lambda blk, k: f"{g[blk][k]:.4f}" if blk in g and g[blk].get(k) is not None else '-'
    rows.append(f"| {os.path.basename(p)[4:-5]} | {r['heldout_bpb']:.4f} | {copy} | {f('true', 'index_right')} | {f('true', 'whole')} | "
                f"{f('own_index', 'whole')} | {f('null_request', 'whole')} | {f('bank_off', 'whole')} | {f('true', 'head_right')} | {r['train_minutes']:.1f} |")
display(Markdown('\n'.join(rows)))